# Parkinson AI — Colab GPU pipeline
## WideResNet50-2 + Focal Loss → DeepRBF → combined inference

This notebook is designed to run on a **Google Colab GPU runtime**, including when opened in VS Code with the official Google Colab extension.

It follows the team's shared-data layout:
- Dataset ZIP: `My Drive/ParkinsonResearch/datasets/parkinson_dataset_v1.zip`
- Extracted working dataset: `/content/ParkinsonData`
- Persistent training outputs: `My Drive/ParkinsonResearch/runs/`
- Model code: cloned from the exact GitHub commit recorded in the setup cell

Training runs in Colab, not on the laptop's CPU/GPU. The notebook refuses to start if it is not connected to a Colab runtime with CUDA.

### Training order

1. Train the fold-specific WideResNet50-2 baseline with **Focal Loss**.
2. Freeze that exact Focal-trained backbone and train DeepRBF.
3. Select rejection thresholds from development validation, export the combined bundle, and evaluate validation.

Start with fold 1. After the pilot works, change `FOLD_END` to 25. The locked test partition is not used in training or threshold selection.

## 0. Connect to Colab, mount the shared Drive, and prepare the code/data

### VS Code connection

1. Install the official **Google Colab** extension in VS Code.
2. Open this `.ipynb` file.
3. Click **Select Kernel** → **Colab** → choose a Colab runtime.
4. Sign in to the Google account that has access to the team's `ParkinsonResearch` Drive folder.
5. Select a runtime with a **GPU**.

The first code cell checks CUDA and stops if the runtime is not a Colab GPU.

### Why the code comes from GitHub

The previous `parkinson_training_code_v2.zip` was made for the baseline notebook. To avoid silently using an older code snapshot, this notebook checks out the current DeepRBF-only source at a fixed Git commit. The shared dataset remains the exact versioned ZIP already used by the team.

The dataset is copied from Drive to Colab's temporary disk, checksum-verified, safely extracted, and every packaged file is verified. Model checkpoints and reports are saved back to Drive so they survive a Colab runtime reset.

In [1]:
from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile

# This notebook must run on a managed Google Colab runtime, not a local kernel.
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError(
        "This notebook must use a Google Colab kernel. In VS Code choose "
        "Select Kernel → Colab, sign in, and connect to a GPU runtime."
    ) from exc

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is attached to this Colab runtime. In the Colab kernel/runtime "
        "selector, choose a GPU runtime, reconnect, and rerun this cell. "
        "Training has NOT started."
    )

# Shared paths used by the team's other Colab notebooks.
drive.mount("/content/drive")

# CHANGED: point directly to the shared folder via its Drive ID, because the
# shortcut in my own My Drive is named "datasets", not "ParkinsonResearch".
DRIVE_DIR = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch"
)
SOURCE_ZIP = DRIVE_DIR / "datasets/parkinson_dataset_v1.zip"
DATA_ROOT = Path("/content/ParkinsonData")
DATASET_DIR = DATA_ROOT / "datasets"
MANIFEST = DATASET_DIR / "annotations/fold_manifest.csv"
RUNS_DIR = DRIVE_DIR / "runs"

DATASET_SHA256 = "85efa52f80adbd357537842146ae107e3cd6b9817eb7ec95beba2644d813f1c6"
REPOSITORY_URL = "https://github.com/Aihan-2005/ParkinsonAiResearch.git"
CODE_COMMIT = "a1d1bfcb3e0234c24a3c9c0c81ece6ace26ab835"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def extract_checked(source: Path, destination: Path, expected_sha256: str) -> None:
    if not source.is_file():
        raise FileNotFoundError(
            f"Required Drive archive is missing: {source}. "
            "Check that the correct Google account is connected and the shared "
            "ParkinsonResearch folder is available."
        )
    local_zip = Path("/content") / source.name
    shutil.copy2(source, local_zip)
    actual_sha256 = sha256_file(local_zip)
    if actual_sha256 != expected_sha256:
        raise ValueError(
            f"SHA-256 mismatch for {source.name}. "
            f"Expected {expected_sha256}; received {actual_sha256}."
        )

    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(local_zip) as archive:
        root = destination.resolve()
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(root):
                raise ValueError(f"Unsafe ZIP path: {member.filename}")
        archive.extractall(destination)


# 0) Load the shared dataset using the exact same version/checksum as the
# team's 00_shared_dataset_colab.ipynb and 01_wideresnet50_2.ipynb.
extract_checked(SOURCE_ZIP, DATA_ROOT, DATASET_SHA256)

metadata_path = DATA_ROOT / "dataset_package.json"
if not metadata_path.is_file():
    raise FileNotFoundError(f"Dataset package metadata is missing: {metadata_path}")
dataset_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
for relative_name, info in dataset_metadata["files"].items():
    target = (DATA_ROOT / relative_name).resolve()
    if not target.is_relative_to(DATA_ROOT.resolve()):
        raise ValueError(f"Unsafe dataset metadata path: {relative_name}")
    if not target.is_file() or sha256_file(target) != info["sha256"]:
        raise ValueError(f"Dataset file failed integrity verification: {relative_name}")

if not MANIFEST.is_file():
    raise FileNotFoundError(f"Shared fold manifest is missing: {MANIFEST}")

# 1) Fetch the exact refactored source commit from GitHub. This commit contains
# the DeepRBF-only implementation and no active CentreRBF model path.
PROJECT_DIR = Path("/content/ParkinsonAiResearch")
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
subprocess.run(
    ["git", "clone", "--quiet", REPOSITORY_URL, str(PROJECT_DIR)],
    check=True,
)
subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "checkout", "--quiet", "--detach", CODE_COMMIT],
    check=True,
)
actual_commit = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"], text=True
).strip()
if actual_commit != CODE_COMMIT:
    raise RuntimeError(
        f"Unexpected code revision: {actual_commit}; expected {CODE_COMMIT}."
    )

required_code = [
    "train_all_folds.py",
    "train_deep_rbf.py",
    "train_deep_rbf_all_folds.py",
    "tune_deep_rbf_thresholds_all_folds.py",
    "export_deep_rbf_bundle.py",
    "model/deep_rbf_system.py",
]
missing_code = [name for name in required_code if not (PROJECT_DIR / name).is_file()]
if missing_code:
    raise FileNotFoundError(f"GitHub code snapshot is incomplete: {missing_code}")

# Compatibility cleanup for the current DeepRBF-only commit:
# reject_distances() no longer accepts the obsolete gap_scope keyword, while
# two utility call sites and the old test still pass it. Remove only that
# obsolete keyword and the test that asserts the retired legacy behavior.
def remove_obsolete_gap_scope(path: Path) -> None:
    source = path.read_text(encoding="utf-8")
    updated = source.replace(', gap_scope="both_inside"', "")
    updated = updated.replace(", gap_scope='both_inside'", "")
    updated = re.sub(
        r"^[ \t]*gap_scope\s*=\s*[\"']both_inside[\"'],?[ \t]*\n",
        "",
        updated,
        flags=re.M,
    )
    if path.name == "test_deep_rbf_thesis.py":
        updated, _ = re.subn(
            r"(?:\A|\n+)def test_legacy_centre_rbf_gap_rule_remains_global_by_default\(\):"
            r"[\s\S]*\Z",
            "",
            updated,
        )
        updated = updated.rstrip() + "\n"
    path.write_text(updated, encoding="utf-8")


for relative in (
    "utils/rbf_pipeline.py",
    "utils/rbf_thresholds.py",
    "tests/test_deep_rbf_thesis.py",
):
    source_path = PROJECT_DIR / relative
    if source_path.is_file():
        remove_obsolete_gap_scope(source_path)

for relative in ("utils/rbf_pipeline.py", "utils/rbf_thresholds.py", "tests/test_deep_rbf_thesis.py"):
    if "gap_scope=" in (PROJECT_DIR / relative).read_text(encoding="utf-8"):
        raise RuntimeError(f"Obsolete gap_scope call remains in {relative}")

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

# Keep Colab's preinstalled CUDA-compatible torch/torchvision. Install only
# the other project dependencies so pip does not replace the GPU PyTorch build.
requirements = [
    line.strip()
    for line in (PROJECT_DIR / "requirements.txt").read_text().splitlines()
    if line.strip() and not line.lstrip().startswith("#")
    and not line.lower().startswith(("torch", "torchvision"))
]
if requirements:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *requirements],
        check=True,
    )

import torchvision
import model.rbf as rbf_module
from model.rbf import DeepRBF, reject_distances

if not hasattr(DeepRBF, "forward") or hasattr(rbf_module, "CentreRBF"):
    raise RuntimeError("The active source is not the expected DeepRBF-only implementation.")
# Fail fast if the updated rejection API cannot be imported/executed.
reject_distances(
    torch.tensor([[0.2, 0.8]], device="cuda"),
    distance_threshold=1.0,
    gap_threshold=0.05,
)

DEVICE = "cuda"
TORCH_DEVICE = torch.device("cuda")
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print("Colab GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| torchvision:", torchvision.__version__)
print("Git commit:", actual_commit)
print("Dataset version:", dataset_metadata.get("dataset_version"))
print("Dataset directory:", DATASET_DIR)
print("Fold manifest:", MANIFEST)
print("Persistent Drive runs:", RUNS_DIR)
print("Ready: training will execute on the Colab GPU.")

Mounted at /content/drive
Colab GPU: Tesla T4
PyTorch: 2.14.1+cu130 | torchvision: 0.29.1+cu130
Git commit: a1d1bfcb3e0234c24a3c9c0c81ece6ace26ab835
Dataset version: parkinson_dataset_v1
Dataset directory: /content/ParkinsonData/datasets
Fold manifest: /content/ParkinsonData/datasets/annotations/fold_manifest.csv
Persistent Drive runs: /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs
Ready: training will execute on the Colab GPU.


## 0A. Preflight — verify the DeepRBF implementation before spending GPU time

This runs the focused DeepRBF architecture, loss, rejection-rule, and combined-bundle tests. Training cells should only be run if this cell passes.

In [2]:
smoke_test_command = [
    sys.executable, "-m", "pytest", "-q",
    "tests/test_deep_rbf_thesis.py",
    "tests/test_deep_rbf_system.py",
]
subprocess.run(smoke_test_command, cwd=PROJECT_DIR, check=True)
print("DeepRBF preflight tests passed. GPU:", torch.cuda.get_device_name(0))

DeepRBF preflight tests passed. GPU: Tesla T4


## 1. Configuration — start with one fold

The baseline output directory matches the team's existing focal-baseline notebook, so a completed matching Focal checkpoint can be resumed/reused rather than trained again. The DeepRBF outputs are kept in a separate Drive folder.

Enter the approximate free Google Drive storage in `DRIVE_FREE_GB`. The conservative estimate below includes one baseline checkpoint and one combined bundle per selected fold, plus a safety margin. For the one-fold pilot, allow at least about 3.1 GB; for all 25 folds, allow about 29.5 GB.

Do not change the training recipe halfway through a run. If you change the recipe, use a new output folder.

In [3]:
# Pilot first. After it succeeds, change FOLD_END to 25.
FOLD_START = 1
FOLD_END = 1

SEED = 42

# Stage 1: WideResNet50-2 + Focal Loss (same settings as the team's baseline notebook)
BASELINE_EPOCHS = 15
BASELINE_BATCH_SIZE = 8
HEAD_LR = 1e-3
BACKBONE_LR = 1e-5
WEIGHT_DECAY = 1e-4
FOCAL_GAMMA = 2.0
FOCAL_ALPHA_MODE = "uniform"
AUGMENTATION = "standard"
SAMPLING = "class_balanced"
ROTATION_DEGREES = 10.0
HFLIP_PROB = 0.5
NOISE_STD = 0.01
CHECKPOINT_EVERY = 5

# Stage 2: DeepRBF
RBF_EPOCHS = 30
RBF_DIM = 64
RBF_DISTANCE = "l1"       # L1 and L2 need separate experiments/thresholds
RBF_DISTANCE_POWER = "norm"
KNOWN_MARGIN = 1.0
REJECT_MARGIN = 1.0
BETA = 1.0
RBF_LR = 1e-3
RBF_WEIGHT_DECAY = 0.0
FEATURE_BATCH_SIZE = 8

# Enter the actual free storage shown in Google Drive. The mounted Drive
# does not reliably expose the user's remaining quota to Python.
DRIVE_FREE_GB = 14.2

BASELINE_RUNS = DRIVE_DIR / "runs" / "baselines_shared_dataset_v1"
BASELINE_DIR = BASELINE_RUNS / "focal" / "checkpoints"
RBF_ROOT = DRIVE_DIR / "runs" / "deeprbf_focal_shared_dataset_v1"

BASELINE_DIR.mkdir(parents=True, exist_ok=True)
RBF_ROOT.mkdir(parents=True, exist_ok=True)

selected_folds = FOLD_END - FOLD_START + 1
if DRIVE_FREE_GB is None:
    raise RuntimeError(
        "Set DRIVE_FREE_GB to the actual free space shown in Google Drive before training."
    )

from utils.storage import check_training_space
baseline_space = check_training_space(
    BASELINE_DIR,
    folds=selected_folds,
    losses=1,
    free_bytes=int(DRIVE_FREE_GB * 1e9),
)
# Reserve additional space for one inference bundle (WideResNet weights) per fold.
combined_bundle_reserve_gb = selected_folds * 0.35
estimated_required_gb = baseline_space["required_gb"] + combined_bundle_reserve_gb
if DRIVE_FREE_GB < estimated_required_gb:
    raise RuntimeError(
        f"Insufficient Drive space: {DRIVE_FREE_GB:.1f} GB entered; "
        f"approximately {estimated_required_gb:.1f} GB is recommended, "
        "including combined inference bundles."
    )

print("Folds:", FOLD_START, "to", FOLD_END)
print("GPU:", torch.cuda.get_device_name(0))
print("Focal checkpoints:", BASELINE_DIR)
print("DeepRBF outputs:", RBF_ROOT)
print(f"Approximate Drive-space budget: {estimated_required_gb:.1f} GB")

Folds: 1 to 1
GPU: Tesla T4
Focal checkpoints: /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_dataset_v1/focal/checkpoints
DeepRBF outputs: /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/deeprbf_focal_shared_dataset_v1
Approximate Drive-space budget: 3.9 GB


## 2. Stage 1 — train WideResNet50-2 with Focal Loss

This uses the same `train_all_folds.py` training script, fold manifest, augmentation settings, optimizer settings, checkpoint cadence, and Drive checkpoint layout as the teammate's `01_wideresnet50_2.ipynb`.

Only the **Focal Loss** experiment is run here; the CE comparison is not needed for the requested DeepRBF pipeline. `--resume` reuses completed matching folds and resumes incomplete ones.

In [4]:
from utils.notebook_training import run_with_progress

baseline_command = [
    sys.executable, "-u", "train_all_folds.py",
    "--fold-start", str(FOLD_START),
    "--fold-end", str(FOLD_END),
    "--loss", "focal",
    "--focal-gamma", str(FOCAL_GAMMA),
    "--focal-alpha-mode", FOCAL_ALPHA_MODE,
    "--augmentation", AUGMENTATION,
    "--rotation-degrees", str(ROTATION_DEGREES),
    "--hflip-prob", str(HFLIP_PROB),
    "--noise-std", str(NOISE_STD),
    "--sampling", SAMPLING,
    "--label-source", "consensus",
    "--pretrained",
    "--freeze-policy", "early",
    "--epochs", str(BASELINE_EPOCHS),
    "--batch-size", str(BASELINE_BATCH_SIZE),
    "--image-size", "224",
    "--num-workers", "0",
    "--lr", str(HEAD_LR),
    "--backbone-lr", str(BACKBONE_LR),
    "--weight-decay", str(WEIGHT_DECAY),
    "--seed", str(SEED),
    "--device", DEVICE,
    "--manifest", str(MANIFEST),
    "--root", str(DATA_ROOT),
    "--checkpoint-dir", str(BASELINE_DIR),
    "--checkpoint-every", str(CHECKPOINT_EVERY),
    "--progress-json",
    "--resume",
]
print(" ".join(baseline_command))
run_with_progress(
    baseline_command,
    title="WideResNet50-2 + Focal Loss",
    fold_start=FOLD_START,
    fold_end=FOLD_END,
    epochs=BASELINE_EPOCHS,
)

/usr/bin/python3 -u train_all_folds.py --fold-start 1 --fold-end 1 --loss focal --focal-gamma 2.0 --focal-alpha-mode uniform --augmentation standard --rotation-degrees 10.0 --hflip-prob 0.5 --noise-std 0.01 --sampling class_balanced --label-source consensus --pretrained --freeze-policy early --epochs 15 --batch-size 8 --image-size 224 --num-workers 0 --lr 0.001 --backbone-lr 1e-05 --weight-decay 0.0001 --seed 42 --device cuda --manifest /content/ParkinsonData/datasets/annotations/fold_manifest.csv --root /content/ParkinsonData --checkpoint-dir /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_dataset_v1/focal/checkpoints --checkpoint-every 5 --progress-json --resume



TLO fold 01/25 | loss=focal | freeze=early | seed=42
Command: /usr/bin/python3 -u /content/ParkinsonAiResearch/train.py --fold-id 1 --architecture wide_resnet50_2 --label-source consensus --loss focal --focal-gamma 2.0 --focal-alpha-mode uniform --augmentation standard --rotation-degrees 10.0 --hflip-prob 0.5 --noise-std 0.01 --sampling class_balanced --freeze-policy early --epochs 15 --batch-size 8 --image-size 224 --num-workers 0 --lr 0.001 --backbone-lr 1e-05 --weight-decay 0.0001 --seed 42 --device cuda --manifest /content/ParkinsonData/datasets/annotations/fold_manifest.csv --root /content/ParkinsonData --checkpoint-dir /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_dataset_v1/focal/checkpoints --checkpoint-every 5 --progress --progress-json --pretrained --resume
Final-epoch validation predictions: /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_datas

## 3. Stage 2 — train DeepRBF on the frozen Focal-trained features

The trainer finds the matching Focal checkpoint for every requested fold and verifies the fold ID, class mapping, 2048-D feature width, and exact manifest hash.

The WideResNet is frozen and used only to extract features. DeepRBF is trained independently; Focal Loss is not applied a second time. `--resume` continues from the last saved DeepRBF checkpoint after a Colab runtime reset.

In [5]:
rbf_command = [
    sys.executable, "-u", "train_deep_rbf_all_folds.py",
    "--baseline-dir", str(BASELINE_DIR),
    "--baseline-glob", "*_last.pt",
    "--manifest", str(MANIFEST),
    "--root", str(DATA_ROOT),
    "--output-root", str(RBF_ROOT),
    "--fold-start", str(FOLD_START),
    "--fold-end", str(FOLD_END),
    "--epochs", str(RBF_EPOCHS),
    "--rbf-dim", str(RBF_DIM),
    "--distance", RBF_DISTANCE,
    "--distance-power", RBF_DISTANCE_POWER,
    "--known-margin", str(KNOWN_MARGIN),
    "--reject-margin", str(REJECT_MARGIN),
    "--beta", str(BETA),
    "--balance-groups",
    "--lr", str(RBF_LR),
    "--weight-decay", str(RBF_WEIGHT_DECAY),
    "--seed", str(SEED),
    "--device", DEVICE,
    "--feature-batch-size", str(FEATURE_BATCH_SIZE),
    "--resume",
]
print(" ".join(rbf_command))
subprocess.run(rbf_command, cwd=PROJECT_DIR, check=True)

/usr/bin/python3 -u train_deep_rbf_all_folds.py --baseline-dir /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_dataset_v1/focal/checkpoints --baseline-glob *_last.pt --manifest /content/ParkinsonData/datasets/annotations/fold_manifest.csv --root /content/ParkinsonData --output-root /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/deeprbf_focal_shared_dataset_v1 --fold-start 1 --fold-end 1 --epochs 30 --rbf-dim 64 --distance l1 --distance-power norm --known-margin 1.0 --reject-margin 1.0 --beta 1.0 --balance-groups --lr 0.001 --weight-decay 0.0 --seed 42 --device cuda --feature-batch-size 8 --resume


CompletedProcess(args=['/usr/bin/python3', '-u', 'train_deep_rbf_all_folds.py', '--baseline-dir', '/content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_dataset_v1/focal/checkpoints', '--baseline-glob', '*_last.pt', '--manifest', '/content/ParkinsonData/datasets/annotations/fold_manifest.csv', '--root', '/content/ParkinsonData', '--output-root', '/content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/deeprbf_focal_shared_dataset_v1', '--fold-start', '1', '--fold-end', '1', '--epochs', '30', '--rbf-dim', '64', '--distance', 'l1', '--distance-power', 'norm', '--known-margin', '1.0', '--reject-margin', '1.0', '--beta', '1.0', '--balance-groups', '--lr', '0.001', '--weight-decay', '0.0', '--seed', '42', '--device', 'cuda', '--feature-batch-size', '8', '--resume'], returncode=0)

## 4. Stage 3A — select rejection thresholds using development validation only

The tuner prefers a feasible threshold selected from each fold's own validation data. Fold 22 has no validation images, so it is skipped for local fitting and receives the pooled out-of-fold development-validation fallback if that fallback is feasible.

This step does not load the locked test partition. A one-fold pilot may not contain enough validation participants for a feasible threshold; if so, the notebook preserves the trained weights and defers bundle export until more development folds are available.

In [6]:
threshold_command = [
    sys.executable, "-u", "tune_deep_rbf_thresholds_all_folds.py",
    "--baseline-dir", str(BASELINE_DIR),
    "--baseline-glob", "*_last.pt",
    "--deep-rbf-root", str(RBF_ROOT),
    "--manifest", str(MANIFEST),
    "--root", str(DATA_ROOT),
    "--fold-start", str(FOLD_START),
    "--fold-end", str(FOLD_END),
    "--distance", RBF_DISTANCE,
    "--distance-power", RBF_DISTANCE_POWER,
    "--selection-unit", "participants",
    "--max-pd-to-control-rate", "0.0",
    "--min-coverage", "0.0",
    "--feature-batch-size", str(FEATURE_BATCH_SIZE),
    "--device", DEVICE,
]
threshold_report_path = RBF_ROOT / "threshold_selection_global" / "threshold_search_report.json"
if threshold_report_path.exists():
    threshold_report_path.unlink()

print(" ".join(threshold_command))
threshold_run = subprocess.run(threshold_command, cwd=PROJECT_DIR, check=False)

if not threshold_report_path.is_file():
    raise RuntimeError(
        f"Threshold tuner failed before writing its report (exit={threshold_run.returncode})."
    )
threshold_report = json.loads(threshold_report_path.read_text(encoding="utf-8"))
THRESHOLDS_READY = threshold_report.get("status") == "selected"
if not THRESHOLDS_READY:
    print(
        "No feasible threshold was found for this fold selection. The trained WideResNet "
        "and DeepRBF checkpoints are preserved in Drive. Do not invent a threshold; "
        "train additional development folds and rerun this cell."
    )
else:
    print("Threshold selection complete. Fold-specific decision files are ready.")

/usr/bin/python3 -u tune_deep_rbf_thresholds_all_folds.py --baseline-dir /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_dataset_v1/focal/checkpoints --baseline-glob *_last.pt --deep-rbf-root /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/deeprbf_focal_shared_dataset_v1 --manifest /content/ParkinsonData/datasets/annotations/fold_manifest.csv --root /content/ParkinsonData --fold-start 1 --fold-end 1 --distance l1 --distance-power norm --selection-unit participants --max-pd-to-control-rate 0.0 --min-coverage 0.0 --feature-batch-size 8 --device cuda
No feasible threshold was found for this fold selection. The trained WideResNet and DeepRBF checkpoints are preserved in Drive. Do not invent a threshold; train additional development folds and rerun this cell.


## 5. Stage 3B — export combined bundles and evaluate validation (when thresholds are feasible)

Each bundle contains:
- the trained Focal WideResNet weights and preprocessing metadata;
- the separately trained DeepRBF weights and configuration;
- the matching validation-selected rejection thresholds;
- provenance hashes tying the stages together.

The combined model uses `WideResNet.forward_features(images) → DeepRBF(features)`. The baseline classifier logits are not averaged with RBF distances.

In [7]:
if THRESHOLDS_READY:
    for fold in range(FOLD_START, FOLD_END + 1):
        fold_name = f"fold{fold:02d}_{RBF_DISTANCE}_{RBF_DISTANCE_POWER}"
        fold_dir = RBF_ROOT / fold_name
        rbf_checkpoint = fold_dir / "last.pt"
        decision_config = fold_dir / "threshold_selection" / "rbf_decision.json"
        if not rbf_checkpoint.is_file():
            raise FileNotFoundError(rbf_checkpoint)
        if not decision_config.is_file():
            raise FileNotFoundError(decision_config)

        bundle_path = fold_dir / "combined_model.pt"
        export_command = [
            sys.executable, "export_deep_rbf_bundle.py",
            "--deep-rbf-checkpoint", str(rbf_checkpoint),
            "--decision-config", str(decision_config),
            "--manifest", str(MANIFEST),
            "--root", str(DATA_ROOT),
            "--output", str(bundle_path),
            "--device", DEVICE,
        ]
        subprocess.run(export_command, cwd=PROJECT_DIR, check=True)

        validation_command = [
            sys.executable, "eval_deep_rbf.py",
            "--checkpoint", str(rbf_checkpoint),
            "--manifest", str(MANIFEST),
            "--root", str(DATA_ROOT),
            "--split", "val",
            "--decision-config", str(decision_config),
            "--feature-batch-size", str(FEATURE_BATCH_SIZE),
            "--device", DEVICE,
            "--output-dir", str(fold_dir / "combined_validation"),
        ]
        subprocess.run(validation_command, cwd=PROJECT_DIR, check=True)

    print("Combined bundles and validation reports are saved under:", RBF_ROOT)
else:
    print("Bundle export/evaluation is deferred until a feasible development threshold exists.")

Bundle export/evaluation is deferred until a feasible development threshold exists.


## 6. Compose the two trained models in Python

This cell demonstrates the actual composed forward pass on one nonempty development validation fold. It reports distances and decisions; it does not use expert or clinical labels as model inputs.

For a full 25-fold run, fold 1 is selected for this demonstration. If you run only fold 22, choose another fold with nonempty validation.

In [8]:
from model.deep_rbf_system import load_deep_rbf_bundle, WideResNetDeepRBF
from utils.rbf_pipeline import load_deep_rbf_checkpoint
from config import PreprocessingConfig
from data.dataloader import SpiralDataset, build_eval_transform
from torch.utils.data import DataLoader

demo_fold = next((f for f in range(FOLD_START, FOLD_END + 1) if f != 22), None)
if demo_fold is None:
    raise ValueError("The selected fold range contains no nonempty validation fold.")

demo_dir = RBF_ROOT / f"fold{demo_fold:02d}_{RBF_DISTANCE}_{RBF_DISTANCE_POWER}"
if THRESHOLDS_READY:
    system, bundle = load_deep_rbf_bundle(
        demo_dir / "combined_model.pt", device=TORCH_DEVICE
    )
    decision = bundle["decision"]
    preprocessing_data = bundle["baseline"]["preprocessing"]
else:
    # Forward-only composition for a pilot whose validation set is too small
    # to select a defensible rejection threshold.
    backbone, rbf_head, rbf_checkpoint = load_deep_rbf_checkpoint(
        demo_dir / "last.pt", MANIFEST, DATA_ROOT, TORCH_DEVICE
    )
    system = WideResNetDeepRBF(backbone, rbf_head).to(TORCH_DEVICE).eval()
    preprocessing_data = rbf_checkpoint["source_baseline"]["preprocessing"]
    decision = None

preprocessing = PreprocessingConfig.from_dict(preprocessing_data["config"])
val_dataset = SpiralDataset(
    manifest_path=MANIFEST,
    root=DATA_ROOT,
    fold_id=demo_fold,
    split="val",
    image_size=preprocessing.image_size,
    label_source="consensus",
    transform=build_eval_transform(preprocessing),
    allow_empty=True,
)
if len(val_dataset) == 0:
    print(f"Fold {demo_fold} has empty validation; select another fold for the demo.")
else:
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)
    batch = next(iter(val_loader))
    with torch.no_grad():
        distances = system(batch["image"].to(TORCH_DEVICE))
    if decision is not None:
        labels, reasons, distances = system.predict(
            batch["image"].to(TORCH_DEVICE),
            distance_threshold=float(decision["distance_threshold"]),
            gap_threshold=float(decision["gap_threshold"]),
        )
        names = {0: "control", 1: "parkinson", 2: "unknown"}
        for i, drawing_id in enumerate(batch["drawing_id"]):
            print({
                "drawing_id": drawing_id,
                "distance_control": float(distances[i, 0].cpu()),
                "distance_parkinson": float(distances[i, 1].cpu()),
                "prediction": names[int(labels[i].cpu())],
                "reason": reasons[i],
            })
    else:
        print("Combined forward pass succeeded. No threshold was selected on this small pilot.")
        for i, drawing_id in enumerate(batch["drawing_id"]):
            print({
                "drawing_id": drawing_id,
                "distance_control": float(distances[i, 0].cpu()),
                "distance_parkinson": float(distances[i, 1].cpu()),
            })

Combined forward pass succeeded. No threshold was selected on this small pilot.
{'drawing_id': 'control::subject 01::1_subject 01_1_0.png', 'distance_control': 12.367048263549805, 'distance_parkinson': 16.708534240722656}
{'drawing_id': 'control::subject 01::1_subject 01_2_0.png', 'distance_control': 7.831226348876953, 'distance_parkinson': 10.985298156738281}
{'drawing_id': 'control::subject 01::1_subject 01_3_0.png', 'distance_control': 9.252937316894531, 'distance_parkinson': 9.3310546875}
{'drawing_id': 'control::subject 01::1_subject 01_4_0.png', 'distance_control': 12.056550979614258, 'distance_parkinson': 13.37991714477539}


## 7. Final checks before any held-out test evaluation

- Confirm that all requested Focal baseline checkpoints exist and their `loss.name` is `focal`.
- Confirm that every fold has a `deep_rbf/.../last.pt`.
- Confirm the global threshold report lists the expected nonempty validation folds; fold 22 should be skipped for threshold fitting.
- Review the selected radius/gap and participant-level validation report.
- Freeze the full protocol before opening the held-out test partition.

The notebook deliberately stops at development validation. Do not select hyperparameters or thresholds from the test partition.

In [9]:

import json
import torch
import pandas as pd
from pathlib import Path

# 1. Inspect Fold 1 dataset
df = pd.read_csv(MANIFEST)

fold = df[
    df["development_model_fold"].astype(str).str.strip() == "1"
].copy()

fold["cv_eligible"] = (
    fold["cv_eligible"].astype(str).str.lower()
    .isin(["true", "1", "yes", "y"])
)

print("===== FOLD 1 DATASET =====")

eligible = fold[fold["cv_eligible"]]

print(
    eligible.groupby(
        ["role", "expert_label", "label_status"]
    ).agg(
        images=("drawing_id", "count"),
        participants=("participant_id", "nunique")
    )
)

# 2. Inspect WideResNet Focal checkpoint
print("\n===== FOCAL CHECKPOINT =====")

baseline_files = list(BASELINE_DIR.glob("*fold01*_last.pt"))

for path in baseline_files:
    ckpt = torch.load(
        path, map_location="cpu", weights_only=True
    )
    print("File:", path.name)
    print("Epoch:", ckpt.get("epoch"))
    print("Fold:", ckpt.get("fold_id"))
    print("Loss config:", ckpt.get("training_config", {}).get("loss"))

# 3. Inspect Focal validation report
print("\n===== FOCAL VALIDATION =====")

report_dir = BASELINE_DIR.parent / "validation_predictions"

for path in report_dir.glob("*fold01*validation_report.json"):
    report = json.loads(path.read_text())
    print(json.dumps(report, indent=2, ensure_ascii=False))

# 4. Inspect DeepRBF checkpoint
print("\n===== DEEPRBF CHECKPOINT =====")

rbf_path = RBF_ROOT / "fold01_l1_norm" / "last.pt"

if rbf_path.is_file():
    rbf = torch.load(
        rbf_path, map_location="cpu", weights_only=True
    )
    print("Epoch:", rbf.get("epoch"))
    print("Requested epochs:", rbf.get("epochs_requested"))
    print("History length:", len(rbf.get("history", [])))
    print("Last history:", rbf.get("history", [])[-1])
else:
    print("DeepRBF checkpoint not found:", rbf_path)

# 5. Inspect threshold report
print("\n===== THRESHOLD REPORT =====")

threshold_path = (
    RBF_ROOT / "threshold_selection_global"
    / "threshold_search_report.json"
)

if threshold_path.is_file():
    result = json.loads(threshold_path.read_text())

    for key in [
        "status",
        "pooled_validation_images",
        "pooled_validation_participants",
        "validation_folds_used",
        "fold_validation_image_counts",
    ]:
        print(key, ":", result.get(key))
else:
    print("Threshold report not found.")


===== FOLD 1 DATASET =====
                                                        images  participants
role        expert_label label_status                                       
test_locked control      finalized_by_control_majority      20             8
            parkinson    finalized_by_unanimous_vote         5             3
            unknown      finalized_as_unknown               11             7
train       control      finalized_by_control_majority      47            22
            parkinson    finalized_by_unanimous_vote        22            15
            unknown      finalized_as_unknown               30            18
validation  control      finalized_by_control_majority       4             1

===== FOCAL CHECKPOINT =====
File: wide_resnet50_2_consensus_focal_alpha-uniform_freeze-early_fold01_seed42_cfgd207ef19_m68332842_last.pt
Epoch: 15
Fold: 1
Loss config: focal

===== FOCAL VALIDATION =====
{
  "status": "ok",
  "fold_id": 1,
  "run_name": "wide_resnet50_2_consens

In [10]:

from pathlib import Path

BASELINE_DIR = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/"
    "ParkinsonResearch/runs/"
    "baselines_shared_dataset_v1/focal/checkpoints"
)

RBF_ROOT = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/"
    "ParkinsonResearch/runs/"
    "deeprbf_focal_shared_dataset_v1"
)

def show_sizes(title, files):
    files = sorted(files, key=lambda p: p.stat().st_size, reverse=True)
    total = sum(p.stat().st_size for p in files)

    print(f"\n===== {title} =====")
    print("File count:", len(files))
    print(f"Total: {total / 1024**3:.3f} GB")

    for p in files[:15]:
        print(f"{p.stat().st_size / 1024**2:8.2f} MB | {p}")

    return total, len(files)

baseline_files = list(BASELINE_DIR.glob("*_last.pt"))
rbf_files = list(RBF_ROOT.glob("fold*/last.pt"))

baseline_size, baseline_count = show_sizes(
    "WIDERESNET CHECKPOINTS", baseline_files
)

rbf_size, rbf_count = show_sizes(
    "DEEPRBF CHECKPOINTS", rbf_files
)

runs_dir = BASELINE_DIR.parents[2]

all_run_files = [
    p for p in runs_dir.rglob("*") if p.is_file()
]

print("\n===== TOTAL RUN STORAGE =====")
print(
    f"{sum(p.stat().st_size for p in all_run_files) / 1024**3:.3f} GB"
)

if baseline_count:
    print("\n===== BASELINE PROJECTION =====")
    print(
        f"Estimated 25-fold baseline size: "
        f"{baseline_size / baseline_count * 25 / 1024**3:.3f} GB"
    )

if rbf_count:
    print("\n===== DEEPRBF PROJECTION =====")
    print(
        f"Estimated 25-fold DeepRBF size: "
        f"{rbf_size / rbf_count * 25 / 1024**3:.3f} GB"
    )



===== WIDERESNET CHECKPOINTS =====
File count: 1
Total: 0.717 GB
  734.00 MB | /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/baselines_shared_dataset_v1/focal/checkpoints/wide_resnet50_2_consensus_focal_alpha-uniform_freeze-early_fold01_seed42_cfgd207ef19_m68332842_last.pt

===== DEEPRBF CHECKPOINTS =====
File count: 1
Total: 0.003 GB
    3.02 MB | /content/drive/.shortcut-targets-by-id/1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/ParkinsonResearch/runs/deeprbf_focal_shared_dataset_v1/fold01_l1_norm/last.pt

===== TOTAL RUN STORAGE =====
0.720 GB

===== BASELINE PROJECTION =====
Estimated 25-fold baseline size: 17.920 GB

===== DEEPRBF PROJECTION =====
Estimated 25-fold DeepRBF size: 0.074 GB


In [12]:

import torch
from pathlib import Path

BASELINE_DIR = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "1UGKWBfKn2fvGjBr39P1v9ndyA5_8A2Gs/"
    "ParkinsonResearch/runs/"
    "baselines_shared_dataset_v1/focal/checkpoints"
)

checkpoint_path = next(
    BASELINE_DIR.glob("*fold01*_last.pt")
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

def tensor_size(obj):
    if isinstance(obj, torch.Tensor):
        return obj.numel() * obj.element_size()

    if isinstance(obj, dict):
        return sum(tensor_size(v) for v in obj.values())

    if isinstance(obj, (list, tuple)):
        return sum(tensor_size(v) for v in obj)

    return 0

print("===== CHECKPOINT BREAKDOWN =====")

for key, value in checkpoint.items():
    size = tensor_size(value)
    if size > 0:
        print(
            f"{key:30s}: "
            f"{size / 1024**2:.2f} MB"
        )

print("\n===== IMPORTANT METADATA =====")
print("Epoch:", checkpoint["epoch"])
print("Fold:", checkpoint["fold_id"])
print("Loss:", checkpoint["loss"]["name"])
print("Architecture:", checkpoint["architecture"])

print("\n===== MODEL ONLY =====")
model_size = tensor_size(
    checkpoint["model_state_dict"]
)
print(f"{model_size / 1024**2:.2f} MB")

print("\n===== OPTIMIZER ONLY =====")
optimizer_size = tensor_size(
    checkpoint["optimizer_state_dict"]
)
print(f"{optimizer_size / 1024**2:.2f} MB")

print("\n===== PROJECTED MODEL STORAGE =====")
print(
    f"25 folds: "
    f"{model_size * 25 / 1024**3:.2f} GB"
)


===== CHECKPOINT BREAKDOWN =====
model_state_dict              : 255.24 MB
optimizer_state_dict          : 478.47 MB
train_loader_generator_state  : 0.00 MB
torch_rng_state               : 0.00 MB
cuda_rng_state_all            : 0.00 MB

===== IMPORTANT METADATA =====
Epoch: 15
Fold: 1
Loss: focal
Architecture: wide_resnet50_2

===== MODEL ONLY =====
255.24 MB

===== OPTIMIZER ONLY =====
478.47 MB

===== PROJECTED MODEL STORAGE =====
25 folds: 6.23 GB
